# 04 🔧 🐸壊し放題実験場

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagioneko/cse-frog/blob/main/notebooks/04_break_your_frog.ipynb)

> 解説は [HANDBOOK](../docs/HANDBOOK.md) と [CONFIG_GUIDE](../docs/CONFIG_GUIDE.md)、全設定は [REFERENCE](../docs/REFERENCE.md) にあります。

In [1]:
# 🐸 の準備(Colab では最初に1回だけインストールします)
try:
    import cse
except ImportError:
    %pip install -q cse-frog
from cse import Frog, END

## 0. 実験台
下のセルの数字を **好きに変えて** 実行してください。何が壊れたかが表でその場で見えます。

In [2]:
DATA = [["右", "右", "下", "左"]] * 3 + [["左", "右", "上", "右"]] * 3
PREFIX = ["左", "右"]

frog = Frog(
    temperature=0.8,
    probability_mode="softmax",
    top_k_edges=3,
    history_boost=0.0,
    refractory_steps=0,
    max_consecutive_repeats=2,
    pair_context_capacity=2048,
    pair_context_boost=1.0,
    context_trace_decay=0.0,
    context_projection_boost=0.0,
    learning_rate=0.1,
    weight_decay=0.9995,
    temporal_learning_window=1,
).learn(DATA)

frog.show(PREFIX)

🐸 ['左', '右'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
上          0.884   0.000   1.000   0.000    1.884   0.637
<END>      0.884   0.000   0.000   0.000    0.884   0.183
下          0.877   0.000   0.000   0.000    0.877   0.181
右          0.877   0.000   0.000   0.000    0.877   0.000


💡 点数があるのに確率が `0.000` の候補は、`top_k_edges=3` で確率の対象から外された候補です。`top_k_edges=5` にしてみてください。

## 1. 「何も変わらない」の診断器
設定を1つ変えて、**1位・確率・点数** のどれが変わったかを調べます。
- 点数が変わったのに確率が同じ → **Type A**(中では変わったが出力に見えない)
- 点数も変わらない → **Type B**(その設定の回路が閉じている = 前提スイッチがOFF)

In [3]:
def diagnose(change, base=None, data=DATA, prefix=PREFIX):
    base = dict(base or {})
    a = Frog(**base).learn(data)
    b = Frog(**{**base, **change}).learn(data)
    same_top = a.predict(prefix) == b.predict(prefix)
    same_prob = a.probabilities(prefix) == b.probabilities(prefix)
    same_score = a.scores(prefix) == b.scores(prefix)
    if not same_prob:
        kind = "変わった(確率が動いた)"
    elif not same_score:
        kind = "Type A: 点数は変わったが確率は同じ"
    else:
        kind = "Type B: 点数も同じ(前提スイッチを疑う)"
    print(f"{str(change):<40} 1位同じ={same_top!s:<5} → {kind}")

diagnose({"temperature": 3.0})
diagnose({"threshold": 0.8})
diagnose({"activation_decay": 0.1})
diagnose({"context_trace_decay": 0.9})
diagnose({"max_consecutive_repeats": 5})

{'temperature': 3.0}                     1位同じ=True  → 変わった(確率が動いた)
{'threshold': 0.8}                       1位同じ=True  → Type B: 点数も同じ(前提スイッチを疑う)
{'activation_decay': 0.1}                1位同じ=True  → Type B: 点数も同じ(前提スイッチを疑う)
{'context_trace_decay': 0.9}             1位同じ=True  → Type B: 点数も同じ(前提スイッチを疑う)
{'max_consecutive_repeats': 5}           1位同じ=True  → Type B: 点数も同じ(前提スイッチを疑う)


## 2. Wake-Up: 前提スイッチを入れると生き返る

In [4]:
print("--- history_boost = 0.35 にすると活性系が生きる ---")
diagnose({"threshold": 0.8}, base={"history_boost": 0.35})
diagnose({"activation_decay": 0.1}, base={"history_boost": 0.35})
diagnose({"top_k_edges": 1}, base={"history_boost": 0.35})
print("--- refractory_steps = 2 にすると例外規則が生きる ---")
diagnose({"max_consecutive_repeats": 5}, base={"refractory_steps": 2}, data=[["A", "A", "A", "B"]] * 4, prefix=["A", "A"])
print("--- temporal_learning_window = 3 にすると距離の設定が生きる ---")
diagnose({"temporal_learning_decay": 1.0}, base={"temporal_learning_window": 3})
diagnose({"direct_learning_window": 1}, base={"temporal_learning_window": 3})

--- history_boost = 0.35 にすると活性系が生きる ---
{'threshold': 0.8}                       1位同じ=True  → 変わった(確率が動いた)
{'activation_decay': 0.1}                1位同じ=False → 変わった(確率が動いた)
{'top_k_edges': 1}                       1位同じ=False → 変わった(確率が動いた)
--- refractory_steps = 2 にすると例外規則が生きる ---
{'max_consecutive_repeats': 5}           1位同じ=False → 変わった(確率が動いた)
--- temporal_learning_window = 3 にすると距離の設定が生きる ---
{'temporal_learning_decay': 1.0}         1位同じ=True  → 変わった(確率が動いた)
{'direct_learning_window': 1}            1位同じ=True  → 変わった(確率が動いた)


## 3. 自由改造
🐸の中身(エンジン)は `frog._engine` にあります。たとえば **top-p(nucleus)** は今の🐸にはありません。自分で足してみましょう。

In [5]:
def top_p(frog, prefix, p=0.9):
    """確率の高い順に足していき、合計が p を超えたところで切る(自作の top-p)。"""
    out, total = [], 0.0
    for sym, prob in frog.top(prefix, k=100):
        out.append((sym, prob))
        total += prob
        if total >= p:
            break
    s = sum(q for _, q in out)
    return [(sym, q / s) for sym, q in out]

data5 = [["A", "B"]] * 5 + [["A", "C"]] * 4 + [["A", "D"]] * 3 + [["A", "E"]] * 2 + [["A", "F"]] * 1
for k in [3, 100]:
    f = Frog(top_k_edges=k).learn(data5)
    print(f"top_k_edges={k:<4} そのまま : {[(s, round(q, 3)) for s, q in f.top(['A'], k=10)]}")
    print(f"{'':16} top-p 0.9: {[(s, round(q, 3)) for s, q in top_p(f, ['A'], 0.9)]}")

top_k_edges=3    そのまま : [('B', 0.484), ('C', 0.314), ('D', 0.202)]
                 top-p 0.9: [('B', 0.484), ('C', 0.314), ('D', 0.202)]
top_k_edges=100  そのまま : [('B', 0.399), ('C', 0.259), ('D', 0.167), ('E', 0.107), ('F', 0.068)]
                 top-p 0.9: [('B', 0.429), ('C', 0.278), ('D', 0.179), ('E', 0.115)]


既定の🐸(`top_k_edges=3`)では、**top-pを足したつもりが何も切れていません**。確率を出す前に、候補がもう3匹に絞られているからです。`top_k_edges=100` にして候補を全部残すと、ようやく top-p らしく「E」までで切れます。設定どうしがつながった回路になっている、もう1つの例です。

壊れたら `Frog()` を作り直せば元どおり。**Tune it. Wake it. Break it. Explain it.** 🐸🎛️🔬